In [15]:
!pip install pyreadstat pandas

In [16]:
import os
import pandas as pd
import pyreadstat
from google.colab import drive
from datetime import datetime

In [17]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
ROOT = "/content/drive/MyDrive/ENARES_2024_PROJECT"

RAW_DIR = os.path.join(ROOT, "01BasesDatosPrimarias")

REPORT_DIR = os.path.join(ROOT, "04CuestionariosInformes/reportes")
LOG_DIR = os.path.join(ROOT, "05Resultados/logs")

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

In [19]:
sav_files = []

for root, _, files in os.walk(RAW_DIR):
    for f in files:
        if f.endswith(".sav"):
            sav_files.append(os.path.join(root, f))

len(sav_files)

22

In [20]:
def extract_metadata(file_path):

    df, meta = pyreadstat.read_sav(file_path)

    module_id = os.path.basename(os.path.dirname(file_path))

    return {
        "file": file_path,
        "module": module_id,
        "n_rows": df.shape[0],
        "n_columns": df.shape[1],
        "columns": list(df.columns),
        "column_labels": meta.column_names_to_labels,
        "value_labels": meta.variable_value_labels,
        "missing_ranges": meta.missing_ranges
    }

In [21]:
metadata_list = []

for f in sav_files:
    try:
        metadata_list.append(extract_metadata(f))
        print("Processed:", f)
    except Exception as e:
        print("Failed:", f, e)

Processed: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1941/extracted/976-Modulo1941/01_CRS01_CAP100.sav
Processed: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1942/extracted/976-Modulo1942/02_CRS01_CAP200.sav
Processed: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1943/extracted/976-Modulo1943/03_CRS01_CAP300.sav
Processed: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1944/extracted/976-Modulo1944/04_CRS01_CAP400.sav
Processed: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1945/extracted/976-Modulo1945/05_CRS01_CAP402.sav
Processed: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1946/extracted/976-Modulo1946/06_CRS01_CAP405.sav
Processed: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1947/extracted/976-Modulo1947/07_CRS01_CAP411.sav
Processed: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/

In [22]:
crs_detection_rows = []

for m in metadata_list:

    file_name = os.path.basename(m["file"]).upper()

    detected_crs = "UNKNOWN"

    if "CRS01" in file_name:
        detected_crs = "CRS01"

    elif "CRS02" in file_name:
        detected_crs = "CRS02"

    elif "CRS03" in file_name:
        detected_crs = "CRS03"

    elif "CRS04" in file_name:
        detected_crs = "CRS04"

    crs_detection_rows.append({
        "module_id": m["module"],
        "sav_file": m["file"],
        "detected_crs": detected_crs,
        "n_rows": m["n_rows"],
        "n_columns": m["n_columns"]
    })

df_crs_detection = pd.DataFrame(crs_detection_rows)

df_crs_detection

,module_id,sav_file,detected_crs,n_rows,n_columns
0,976-Modulo1941,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,49
1,976-Modulo1942,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,50285,41
2,976-Modulo1943,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,110
3,976-Modulo1944,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,173
4,976-Modulo1945,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,564
5,976-Modulo1946,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,412
6,976-Modulo1947,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,4264
7,976-Modulo1948,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,97
8,976-Modulo1949,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,36
9,976-Modulo1950,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,CRS01,13826,22


In [23]:
keywords = [
     # violence core
    "violencia", "violento", "golpe", "golpear", "agresión", "agresion",
    "insulto", "humillación", "amenaza", "maltrato", "abuso",

    # sexual violence
    "sexual", "tocamiento", "acoso", "violación", "violacion",

    # school context
    "escuela", "colegio", "aula", "profesor", "clase", "institución educativa",

    # family context
    "familia", "hogar", "padre", "madre", "hermano", "tutor", "casa",

    # social support / relationships
    "apoyo", "redes", "amigos", "amistades", "relación", "relaciones",

    # gender roles / perception
    "género", "genero", "roles", "actitudes", "creencias",

    # adolescent context (soft signals)
    "adolescente", "estudiante", "joven"
]

crs04_results = []

for m in metadata_list:

    file_name = m["file"].upper()
    columns = m["columns"]
    labels = m["column_labels"]

    # ---- 1. filename signal ----
    filename_score = 1 if "CRS04" in file_name else 0

    # ---- 2. label-based semantic score ----
    label_text = " ".join(
        [str(labels.get(col, "")) for col in columns]
    ).lower()

    keyword_score = sum(label_text.count(k) for k in keywords)

    # ---- 3. structure signal ----
    n_rows = m["n_rows"]
    n_cols = m["n_columns"]

    structure_score = 1 if n_rows == 18807 else 0  # CRS04 expected 18807

    # ---- FINAL SCORE ----
    total_score = filename_score + keyword_score + structure_score

    crs04_results.append({
        "module_id": m["module"],
        "sav_file": m["file"],
        "n_rows": n_rows,
        "n_columns": n_cols,
        "filename_score": filename_score,
        "keyword_score": keyword_score,
        "structure_score": structure_score,
        "total_score": total_score
    })

df_crs04_scores = pd.DataFrame(crs04_results)
df_crs04_scores.sort_values("total_score", ascending=False)

,module_id,sav_file,n_rows,n_columns,filename_score,keyword_score,structure_score,total_score
6,976-Modulo1947,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,13826,4264,0,2045,0,2045
20,976-Modulo1961,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,18807,578,1,521,1,523
19,976-Modulo1960,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,18807,523,1,366,1,368
16,976-Modulo1957,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,12904,513,0,348,0,348
4,976-Modulo1945,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,13826,564,0,170,0,170
5,976-Modulo1946,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,13826,412,0,122,0,122
3,976-Modulo1944,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,13826,173,0,109,0,109
18,976-Modulo1959,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,18807,147,1,69,1,71
15,976-Modulo1956,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,12904,119,0,65,0,65
13,976-Modulo1954,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,8955,73,0,62,0,62


In [24]:
df_crs04_candidates = df_crs04_scores[
    df_crs04_scores["sav_file"].str.contains("CRS04", case=False)
]
df_crs04_candidates = df_crs04_candidates.sort_values(
    "total_score",
    ascending=False
)
CRS04_FILES = df_crs04_candidates["sav_file"].tolist()
CRS04_MODULES = df_crs04_candidates["module_id"].tolist()

print("CRS04 modules found:", CRS04_MODULES)
print("Number of CRS04 files:", len(CRS04_FILES))

CRS04 modules found: ['976-Modulo1961', '976-Modulo1960', '976-Modulo1959', '976-Modulo1962']
Number of CRS04 files: 4


In [25]:
df_vars = []
df_values = []
df_missing = []

for m in metadata_list:

    if m["file"] not in CRS04_FILES:
        continue

    for var in m["columns"]:
        df_vars.append({
            "variable_name": var,
            "variable_label": m["column_labels"].get(var),
            "variable_type": "unknown",
            "source_module": m["module"],
            "source_file": m["file"]
        })

    for var, labels in m["value_labels"].items():
        if isinstance(labels, dict):
            for value, label in labels.items():
                df_values.append({
                    "variable_name": var,
                    "value": value,
                    "value_label": label,
                    "source_module": m["module"],
                    "source_file": m["file"]
                })

    if m["missing_ranges"] is not None:
        for var, miss in m["missing_ranges"].items():
            df_missing.append({
                "variable_name": var,
                "missing_code": str(miss),
                "missing_label_or_type": "SPSS_missing",
                "source_module": m["module"],
                "source_file": m["file"],
                "notes": "Extracted via pyreadstat missing_ranges"
            })

df_vars = pd.DataFrame(df_vars)
df_values = pd.DataFrame(df_values)
df_missing = pd.DataFrame(df_missing)

In [26]:
df_vars.to_csv(
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_variables_stage1.csv"),
    index=False
)

df_values.to_csv(
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_value_labels_stage1.csv"),
    index=False
)

if not df_missing.empty:
    df_missing.to_csv(
        os.path.join(LOG_DIR, "ENARES_2024_CRS04_missing_codes_stage1.csv"),
        index=False
    )
else:
    print("No missing codes found for CRS04 — file not created .")

No missing codes found for CRS04 — file not created .


In [27]:
validation_rows = []

for m in metadata_list:
    if m["file"] not in CRS04_FILES:
        continue

    cols = m["columns"]
    text = " ".join(cols).lower()

    age_vars = [v for v in cols if "edad" in v.lower()]
    sex_vars = [v for v in cols if "sexo" in v.lower()]

    # disability range: C4P130_1 to C4P130_6
    disability_vars = [
        v for v in cols
        if v.lower().startswith("c4p130_")
    ]

    weight_vars = [
        v for v in cols
        if any(k in v.lower() for k in ["factor_alumnos"])
    ]

    strata_vars = [
        v for v in cols
        if any(k in v.lower() for k in ["ccdd"])
    ]

    cluster_vars = [
        v for v in cols
        if any(k in v.lower() for k in ["id"])
    ]

    validation_rows.append({
        "module_id": m["module"],
        "sav_file": m["file"],
        "n_rows": m["n_rows"],
        "n_columns": m["n_columns"],

        # now EVIDENCE instead of boolean
        "possible_age_variables": age_vars,
        "possible_sex_variables": sex_vars,
        "possible_disability_variables": disability_vars,
        "possible_weight_variables": weight_vars,
        "possible_strata_variables": strata_vars,
        "possible_cluster_variables": cluster_vars,

        "notes": "CRS04 validation with explicit variable evidence extraction"
    })

df_validation = pd.DataFrame(validation_rows)

df_validation.to_csv(
    os.path.join(LOG_DIR, "ENARES_2024_CRS04_validacion_stage1.csv"),
    index=False
)

In [28]:
report_path = os.path.join(
    REPORT_DIR,
    "ENARES_2024_CRS_identificacion_modulos.md"
)

with open(report_path, "w") as f:

    f.write("# CRS Module Identification Report\n\n")

    f.write("## CRS04 Definition (INEI)\n")
    f.write("Adolescents (12–17 years) and violence (psychological, physical, sexual) in family and school contexts.\n\n")

    f.write("## CRS04 Selection Evidence\n")
    f.write("- Filename contains CRS04 (primary evidence)\n")
    f.write("- SPSS metadata keyword scoring (validation)\n")
    f.write("- Variable structure consistent with violence/school/family theme\n\n")

    f.write("## Selected CRS04 Module\n")
    f.write(f"- Module: {CRS04_MODULES}\n")
    f.write(f"- File: {CRS04_FILES}\n")

print("Report saved:", report_path)

Report saved: /content/drive/MyDrive/ENARES_2024_PROJECT/04CuestionariosInformes/reportes/ENARES_2024_CRS_identificacion_modulos.md
